# Popularitas
        **recsys › top_n** · notebook `popularity`

        **Kapan dipakai:** Baseline top-N wajib dan sering sulit dikalahkan: rekomendasikan film paling populer yang belum ditonton.

        **Jebakan umum:** Tidak personal; cakupan katalog sangat rendah (itu-itu saja).

**Data demo:** MovieLens 100k (943 pengguna, 1682 film, 100.000 rating). Umpan balik implisit (pengguna menilai = menonton). Evaluasi **leave-last-out**: interaksi terakhir tiap pengguna disembunyikan; model harus menempatkannya setinggi mungkin di antara SEMUA film yang belum ditonton.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
DATA = "data_latihan/recsys/ml-100k"; MODEL_KEY = "popularity"; K = 10
OUT = out_dir("recsys", "top_n", MODEL_KEY); set_seed(42)

## 2. Data dan split leave-last-out

In [ ]:
from types import SimpleNamespace
from superdb.recsys import load_ml100k, temporal_split, leave_last_out, interaction_matrix, rating_metrics, rank_metrics
r, movies, users, n_users, n_items = load_ml100k(DATA)
genre_cols = [c for c in movies.columns if c not in ("item", "title", "release_date", "i", "unknown")]
ctx = SimpleNamespace(n_users=n_users, n_items=n_items, movies=movies, users=users, genres=genre_cols, genre_cols=genre_cols)
print(f"{len(r)} rating | {n_users} pengguna x {n_items} film | kerapatan {len(r) / (n_users * n_items):.1%} | rata-rata rating {r.rating.mean():.2f}")

In [ ]:
train, last = leave_last_out(r); R = interaction_matrix(train, n_users, n_items); test_items = last.sort_values("u").i.to_numpy()
print(f"latih {len(train)} interaksi | uji: {len(last)} (satu per pengguna) | kerapatan matriks {R.nnz / np.prod(R.shape):.1%}")

## 3. Model
Bentuk seragam: `score_matrix(R, ctx)` mengembalikan skor (pengguna × film); film yang sudah ditonton otomatis dikecualikan saat evaluasi.

In [ ]:
import numpy as np, scipy.sparse as sp
from superdb.recsys import item_knn_scores, user_knn_scores

def score_matrix(R, ctx):
    return np.tile(np.asarray(R.sum(0)).ravel(), (R.shape[0], 1))

## 4. Evaluasi
**HR@10** (hit rate): porsi pengguna yang film uji-nya masuk 10 teratas. **NDCG@10** & **MRR**: memperhitungkan posisi. **Coverage**: variasi film yang pernah direkomendasikan (rekomendasi yang sama untuk semua = coverage kecil). Pembanding acak: HR@10 ≈ 10/1682 ≈ 0.6%.

In [ ]:
import time; t0 = time.time(); S = np.asarray(score_matrix(R, ctx), np.float32); sec = time.time() - t0
assert S.shape == (n_users, n_items) and np.isfinite(S).all()
metrics = rank_metrics(S, R, test_items, K); metrics["seconds"] = sec; display(show_table(metrics)); print(f"waktu: {sec:.1f} dtk")

## 5. Contoh rekomendasi untuk satu pengguna

In [ ]:
u = 0; Su = S[u].copy(); Su[R[u].nonzero()[1]] = -np.inf; top = np.argsort(-Su)[:5]
seen = movies.set_index("i").loc[train[train.u == u].sort_values("rating", ascending=False).i.head(5), "title"].tolist()
print("film yang disukai pengguna ini:", seen); print("rekomendasi:", movies.set_index("i").loc[top, "title"].tolist()); print("film uji (disembunyikan):", movies.set_index("i").loc[test_items[u], "title"])
pd.DataFrame({"user": np.repeat(users.user.to_numpy(), 5), "item_rec": movies.set_index("i").loc[np.argsort(-np.where(R.toarray() > 0, -np.inf, S), axis=1)[:, :5].ravel(), "item"].to_numpy()}).to_csv(OUT / "top5.csv", index=False)
save_json({"metrics": metrics}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan semua metode: popularitas sering sangat sulit dikalahkan pada data kecil; bandingkan juga coverage.
- Pada data lomba, ganti loader dan pastikan split mengikuti cara test dibuat (waktu/leave-one-out).
- Hibrida (kolaboratif + konten) membantu film baru (cold-start).